<h1 align="center">Laboratorio 8</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/labs/lab8)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert lab8.ipynb --to html

**DataSet:** <https://github.com/eg4000/SKU110K_CVPR19>

Usted forma parte del equipo de Inteligencia Artificial de VisorShelf, una startup guatemalteca de tecnología retail que desarrolla un sistema de auditoría automática de anaqueles para supermercados y tiendas de conveniencia. El sistema debe identificar y localizar productos en imágenes tomadas por cámaras fijas instaladas frente a los anaqueles, con el objetivo de detectar quiebres de stock, productos mal ubicados y desorden en la exhibición.

La restricción operativa principal: las cámaras capturan imágenes cada 30 segundos y el sistema debe procesar cada imagen en menos de $500 , ms$ corriendo on-premise en hardware de tienda (CPU de gama media, sin GPU dedicada). Su equipo dispone de un dataset anotado con bounding boxes de productos en anaquel.

## Task 1

El siguiente conjunto de preguntas evalúa su capacidad de analizar, justificar y conectar los fundamentos matemáticos de la detección de objetos con decisiones de ingeniería reales dentro del proyecto VisorShelf. No basta con enunciar fórmulas: se espera que usted explique el significado de cada término y argumente su relevancia práctica en el contexto de auditoría de anaqueles.

### Pregunta 1.1

El gerente de producto de VisorShelf le presenta la siguiente situación: el sistema detecta una lata de atún en el anaquel y devuelve la caja predicha
$b' = (142, 89, 218, 165)$ en formato $(x_{min}, y_{min}, x_{max}, y_{max})$.

El radiólogo de calidad del cliente anota manualmente la caja real
$b^* = (138, 84, 222, 170)$.

El cliente pregunta: “¿Qué tan buena es esa detección?”

Con esto en mente, responda las siguientes preguntas en su reporte:

#### Inciso 1

Calcule manualmente el IoU entre las dos cajas. Muestre paso a paso el cálculo del área de intersección, el área de unión y el valor final. Explique en términos no técnicos qué significa ese número para el cliente de VisorShelf.

**Respuesta:**

El primer paso es encontrar la región de intersección, es decir, el rectángulo que comparten ambas cajas. Las coordenadas de esa región se calculan tomando el mayor de los mínimos y el menor de los máximos:

$$x_{min}^I = \max(142, 138) = 142, \quad x_{max}^I = \min(218, 222) = 218$$

$$y_{min}^I = \max(89, 84) = 89, \quad y_{max}^I = \min(165, 170) = 165$$

Con esto, el área de intersección es:

$$|I| = (218 - 142) \times (165 - 89) = 76 \times 76 = 5{,}776 \text{ px}^2$$

El área individual de cada caja es:

$$|b'| = (218 - 142) \times (165 - 89) = 76 \times 76 = 5{,}776 \text{ px}^2$$

$$|b^*| = (222 - 138) \times (170 - 84) = 84 \times 86 = 7{,}224 \text{ px}^2$$

El área de unión, por el principio de inclusión-exclusión:

$$|U| = 5{,}776 + 7{,}224 - 5{,}776 = 7{,}224 \text{ px}^2$$

$$IoU = \frac{5{,}776}{7{,}224} \approx 0.7996 \approx 80\%$$

El sistema identificó la lata de atún con un 80% de coincidencia respecto a la ubicación real. El modelo dibujó su cuadro casi encima del cuadro correcto: 8 de cada 10 píxeles de la zona real están cubiertos. Es decir que el sistema no solo encontró el producto, sino que lo ubicó con muy buena precisión.

#### Inciso 2

En la fórmula $IoU = \frac{|I|}{|U|}$, identifique qué representa cada símbolo ($|I|$, $|U|$) y explique por qué el denominador es la unión y no el área del ground truth. ¿Qué problema concreto evita esa decisión de diseño?


**Respuesta:**

- $|I|$ es el **área de intersección**: los píxeles que comparten la caja predicha y la caja real simultáneamente. Representa la zona donde el modelo y el anotador humano coinciden.
- $|U|$ es el **área de unión**: todos los píxeles que cubre cualquiera de las dos cajas, sin contar dos veces los comunes. Es la cobertura total del par de detecciones.

El denominador es la unión y no el área del ground truth; si se usara solo el área de la caja real como denominador, un modelo que predijera una caja enorme que contenga completamente al objeto real obtendría un IoU perfecto de 1.0, aunque estuviera marcando el doble del espacio. La penalización al exceso es lo que le da sentido a la métrica. Al normalizar sobre la unión, cualquier área predicha que sobresalga de la real reduce el valor del IoU proporcionalmente, forzando al modelo a ser preciso y no simplemente grande.

#### Inciso 3

El equipo de VisorShelf está evaluando dos umbrales de IoU para decidir si una detección es válida:
$\theta = 0.5$ y $\theta = 0.75$.

¿Cuál recomendaría para el sistema de auditoría de anaqueles y por qué? Considere el impacto operativo de los falsos positivos y falsos negativos en el negocio del cliente.

**Respuesta:**

Lo que importa es que el sistema confirme que un producto está o no está en su posición, no que mida su posición con precisión quirúrgica. Un IoU de 0.5 significa que la caja predicha cubre al menos la mitad de la zona real del producto, suficiente para saber que el algoritmo encontró ese artículo y puede razonar sobre su presencia.

Usar $\theta = 0.75$ aumenta los falsos negativos. Productos correctamente identificados serían descartados por no alcanzar esa precisión geométrica estricta, generando alertas falsas de quiebre de stock que enviarían al personal a reponer un anaquel que en realidad está surtido.

El umbral de $0.75$ tiene sentido en aplicaciones donde la localización exacta importa como robótica con movimientos precisos


### Pregunta 1.2

Durante una prueba piloto en una tienda de conveniencia, el detector de VisorShelf analiza una imagen con 15 productos en el anaquel. El modelo genera 18 predicciones. Tras aplicar el umbral $IoU = 0.5$, el equipo clasifica:
12 TP, 6 FP y 3 FN.

Con base a esto, responda dentro de su reporte:

#### Inciso 4

Calcule la Precisión y el Recall para esta prueba. En las fórmulas

$P = \frac{TP}{TP + FP}$
y
$R = \frac{TP}{TP + FN}$,

explique verbalmente qué mide cada término del denominador y por qué ambas métricas son necesarias para evaluar el sistema.

**Respuesta:**

Datos: 
- $TP = 12$
- $FP = 6$
- $FN = 3$.

$$P = \frac{12}{12 + 6} = \frac{12}{18} \approx 0.667 \quad (66.7\%)$$

$$R = \frac{12}{12 + 3} = \frac{12}{15} = 0.800 \quad (80.0\%)$$

En el denominador de la **Precisión**, el término $FP$ representa las detecciones que el modelo activó sin que hubiera un producto rea.

En el denominador del **Recall**, el término $FN$ representa los productos reales que el modelo no detectó, los que pasaron invisibles.

Un detector que marque todo como producto alcanzaría recall perfecto pero precisión cercana a cero. El sistema de VisorShelf necesita las dos, perderse un quiebre de stock tiene un costo de negocio directo, pero generar decenas de falsas alarmas por imagen haría inoperable el sistema para el personal de tienda.

#### Inciso 5

El director de operaciones de la tienda le dice: “Prefiero que el sistema no se pierda ningún quiebre de stock, aunque a veces nos avise de falsas alarmas.” Traduzca esa preferencia a términos de Precisión y Recall. ¿Qué umbral de confianza ajustaría y en qué dirección?

**Respuesta:**

Quiere decir que desea bajar el umbral de confianza. Ese umbral es el filtro que decide a partir de qué score el modelo reporta una detección como válida. Al reducirlo, el sistema acepta predicciones menos seguras, lo que aumenta el número de detecciones totales. Eso hace más difícil que un producto real escape sin ser detectado (sube el Recall), pero a cambio se cuelan más predicciones incorrectas (baja la Precisión).

La lógica del negocio indica que un quiebre de stock no detectado implica ventas perdidas y clientes que no encuentran el producto; una falsa alarma implica que un empleado revisa el anaquel y lo encuentra correcto, un costo operativo menor.

#### Inciso 6

Explique qué es el mAP (Mean Average Precision) y por qué es más informativo que reportar un único valor de Precisión o Recall. En su explicación, distinga entre
$mAP@0.5$ (protocolo PASCAL VOC) y
$mAP@0.5:0.95$ (protocolo COCO),
y argumente cuál protocolo sería más exigente para VisorShelf y por qué.

**Respuesta:**

Reportar un único valor de Precisión o Recall es problemático porque ambos dependen de un umbral de confianza fijo y arbitrario. Si el umbral cambia, los valores cambian, y un punto específico no representa el comportamiento general del modelo.

El **mAP (Mean Average Precision)** resuelve esto evaluando el modelo en todos los umbrales de confianza posibles. Para cada clase, se construye una curva Precision-Recall que muestra el trade-off completo a lo largo de todos los posibles umbrales, y el **AP (Average Precision)** de esa clase es el área bajo esa curva. El mAP promedia ese AP sobre todas las clases:

$$mAP = \frac{1}{C} \sum_{c=1}^C AP_c$$

Esto hace que la métrica sea robusta al umbral y representativa del comportamiento global del detector.

**Diferencia entre protocolos:**

- **$mAP@0.5$ (PASCAL VOC):** se calcula con un único umbral de IoU fijo en 0.5. Una detección es TP si su solapamiento con el ground truth supera el 50%. Es el protocolo histórico, más permisivo en cuanto a localización.

- **$mAP@0.5:0.95$ (COCO):** promedia el mAP calculado a 10 umbrales de IoU distintos: $\{0.50, 0.55, 0.60, \ldots, 0.95\}$. Un modelo debe mantener buen desempeño incluso cuando la caja necesita encajar con hasta el 95% de solapamiento.

$$mAP_{COCO} = \frac{1}{10} \sum_{t \in \{0.50,\ 0.55,\ \ldots,\ 0.95\}} mAP@t$$

El protocolo COCO sería más exigente. Primero, los anaqueles son escenas densamente pobladas con productos similares muy juntos: un modelo que localiza bien pero no con precisión estricta sería penalizado más severamente por COCO, lo cual es informativo para saber si el sistema puede distinguir un producto de su vecino inmediato. Segundo, COCO también reporta métricas separadas para objetos pequeños ($AP_S$), relevantes en anaquel donde productos delgados o de bajo perfil pueden ser pequeños en imagen.

### Pregunta 1.3

En una imagen de anaquel con 40 productos, el modelo de VisorShelf genera 312 cajas candidatas antes de cualquier postprocesamiento. El cliente observa el resultado intermedio y exclama: “¡El sistema está viendo el mismo producto decenas de veces!”

Responda lo siguiente en su reporte:

#### Inciso 7

Explique al cliente qué es el Non-Maximum Suppression (NMS) y por qué el detector genera múltiples cajas para el mismo objeto. Describa el algoritmo paso a paso en lenguaje no técnico.

**Respuesta:**

Los detectores no analizan la imagen una sola vez buscando exactamente dónde está cada objeto. En cambio, generan sistemáticamente cientos de cajas candidatas en distintas posiciones, escalas y proporciones sobre la imagen, y para cada una calculan qué tan probable es que contenga un producto. Un objeto real, como una lata de atún, activa docenas de esas cajas candidatas vecinas porque todas ellas se solapan con la región donde está el producto y el modelo les asigna alta confianza.

El **Non-Maximum Suppression (NMS)** es el algoritmo que limpia ese ruido y convierte las cientos de candidatas en detecciones únicas:

1. **Ordenar por confianza**: tomar primero la caja que el modelo considera más segura.
2. **Seleccionar la mejor**: agregarla a la lista de detecciones finales.
3. **Eliminar las que se solapan demasiado**: calcular cuánto se superpone cada caja restante con la seleccionada. Si el solapamiento supera el umbral $\theta_{NMS}$, esa caja se descarta porque probablemente está marcando el mismo objeto.
4. **Repetir**: tomar la siguiente candidata con mayor confianza y volver al paso 3, hasta que no queden cajas.

El resultado final son detecciones donde cada producto aparece marcado exactamente una vez, con la caja que el modelo consideró más precisa.

#### Inciso 8

El parámetro $\theta_{NMS}$ controla qué tan agresivo es el NMS al suprimir cajas. En un anaquel densamente poblado donde los productos están muy juntos, ¿qué valor de $\theta_{NMS}$ recomendaría (alto o bajo) y por qué? Argumente el riesgo en cada dirección.

**Respuesta:**

En un anaquel densamente poblado donde los productos están muy juntos, la recomendación es usar un valor de $\theta_{NMS}$ **alto**.

$\theta_{NMS}$ es el umbral de solapamiento a partir del cual NMS considera que dos cajas están detectando el mismo objeto y elimina la de menor confianza. En un anaquel denso, dos productos legítimamente distintos y adyacentes tendrán sus bounding boxes con solapamiento natural relativamente alto porque están uno al lado del otro.

**Riesgo de $\theta_{NMS}$ bajo** elimina cajas que corresponden a productos distintos porque sus bounding boxes se tocan o se superponen ligeramente. El resultado es que productos reales desaparecen de la detección, generando falsos negativos. En el contexto de VisorShelf, esto significaría no detectar quiebres de stock en productos adyacentes, el peor fallo posible para el negocio.

**Riesgo de $\theta_{NMS}$ alto** permite que sobrevivan cajas que marcaron el mismo objeto desde posiciones ligeramente distintas, generando detecciones duplicadas que inflan artificialmente el conteo de artículos en el anaquel y pueden producir errores en el inventario automatizado.

Un valor alto protege los productos adyacentes a costa de aceptar algunas duplicaciones, que pueden filtrarse con umbrales adicionales de confianza.

#### Inciso 9

¿En qué orden se deben aplicar el umbral de confianza y el NMS? Justifique la respuesta y explique qué sucedería computacionalmente si se invierte ese orden en un sistema que procesa 30 imágenes por minuto.

**Respuesta:**

Primero el umbral de confianza, luego NMS.

El NMS tiene costo $O(n^2)$: debe comparar cada caja contra todas las demás para calcular sus solapamientos. Si se aplica NMS directamente sobre las 312 cajas candidatas, se realizan del orden de $312^2 / 2 \approx 48{,}000$ comparaciones de IoU por imagen.

El umbral de confianza filtra primero todas las cajas con score bajo, reduciendo drásticamente el conjunto de entrada al NMS. Si ese filtro inicial elimina el 80% de las candidatas, NMS solo debe procesar aproximadamente 62 cajas, reduciendo las comparaciones a unas $\sim 1{,}900$, más de 25 veces menos trabajo.

Si se invierte el orden en un sistema que procesa 30 imágenes por minuto. Con 312 cajas por imagen y 30 imágenes por minuto, NMS debe manejar $312 \times 30 = 9{,}360$ cajas por minuto. Aplicando primero el filtro de confianza y quedando solo el 20%, serían $\approx 1{,}860$ cajas por minuto. La diferencia crece cuadráticamente con el número de cajas, no linealmente. En hardware de tienda sin GPU dedicada, esa diferencia puede ser exactamente la que determina si el sistema cumple o no la restricción de latencia de 500 ms por imagen.

## Task 2

Las siguientes preguntas evalúan su comprensión estratégica de la evolución de los detectores de dos etapas y su capacidad de tomar decisiones de arquitectura justificadas dentro del contexto operativo de VisorShelf. Se valorará la coherencia del argumento con las restricciones reales del sistema.

### Pregunta 2.1

El CTO de VisorShelf propone usar el detector original R-CNN (2014) para la primera versión del sistema. El equipo de ingeniería calcula que con el dataset actual y una CPU de tienda, cada imagen tardaría aproximadamente 45 segundos en procesarse.

Con esto responda en su reporte:

#### Inciso 1

Identifique el cuello de botella principal de R-CNN que causa esa latencia. Explique por qué procesar 2,000 propuestas de región de forma independiente es computacionalmente costoso, conectando su respuesta con lo que la red hace internamente en cada pasada.

**Respuesta:**

El cuello de botella de R-CNN es que procesa cada una de las ~2,000 propuestas de región de forma completamente independiente a través de la CNN. Cada propuesta se recorta de la imagen original, se redimensiona a $227 \times 227$ px y se pasa por completo a través del backbone convolucional para producir un vector de 4,096 dimensiones. Eso significa que la CNN se ejecuta 2,000 veces por imagen.

El problema no es solo el número de ejecuciones, sino la redundancia masiva de cómputo que implica. Las propuestas generadas por Selective Search se solapan frecuentemente entre sí.

#### Inciso 2

Fast R-CNN introdujo el feature map compartido y el RoI Pooling para resolver ese cuello de botella. Explique la lógica detrás de cada uno, ¿qué cómputo elimina el feature map compartido?, ¿qué problema resuelve el RoI Pooling y qué operación matemática realiza para producir un tensor de tamaño fijo a partir de regiones de tamaño variable?

**Respuesta:**

**Feature map compartido:** Fast R-CNN invierte el orden del proceso. En lugar de recortar la imagen primero y pasar cada recorte por la CNN, pasa la imagen completa una sola vez por el backbone y obtiene un único feature map compartido.

Lo que elimina es la redundancia identificada antes, es decir los píxeles que aparecen en múltiples propuestas son procesados por las convoluciones una única vez. El tiempo del forward pass de CNN se reduce.

**RoI Pooling:** El problema que resuelve es que las regiones proyectadas sobre el feature map tienen tamaños variables en celdas (una propuesta puede ocupar $8 \times 12$ celdas, otra $20 \times 7$), pero el clasificador que viene después requiere un vector de dimensión fija. El RoI Pooling toma una región rectangular de tamaño arbitrario en el feature map, la divide en una grilla fija de $H \times W$ celdas y aplica max-pooling dentro de cada celda. El resultado es siempre un tensor de $H \times W \times C$ independientemente del tamaño original de la región, permitiendo que todas las propuestas pasen por el mismo clasificador sin redimensionado de la imagen original.

#### Inciso 3

Con Fast R-CNN el tiempo de CNN bajó a $0.3 , s$/imagen, pero el tiempo total seguía siendo $\sim 2.3 , s$. Identifique el nuevo cuello de botella y explique por qué Selective Search representa un problema arquitectónico más profundo que simplemente ser lento.

**Respuesta:**

Con Fast R-CNN, el cuello de botella se desplaza hacia **Selective Search**, el algoritmo externo que genera las propuestas de región. Su tiempo de ejecución es de ~1.5 segundos por imagen corriendo en CPU, lo que domina el tiempo total de ~2.3 segundos cuando la CNN tarda solo 0.14 segundos.

Selective Search es un algoritmo clásico basado en agrupación de superpíxeles por color, textura y forma. No aprende nada de los datos. Sus propuestas son ciegas al contenido semántico de la imagen; no sabe qué es un producto de anaquel y qué es el fondo, por lo que genera propuestas con el mismo criterio para cualquier imagen. Esto significa que no puede mejorar con más datos de entrenamiento, no puede adaptarse al dominio específico de VisorShelf. El sistema no es entrenable de extremo a extremo, lo que limita estructuralmente la precisión final alcanzable independientemente de qué tan bueno sea el clasificador.

### Pregunta 2.2

El equipo de VisorShelf decide usar Faster R-CNN como base del sistema. Un ingeniero junior pregunta:
“¿Por qué necesitamos una Region Proposal Network si ya tenemos el feature map? ¿No podríamos simplemente hacer sliding window directamente sobre el feature map?”

Responda en su reporte:

#### Inciso 4

Responda la pregunta del ingeniero junior. Explique qué hace la RPN que un sliding window clásico no puede hacer, y por qué el hecho de que la RPN opere sobre el mismo feature map del backbone es una ventaja semántica, no solo computacional.

**Respuesta:**

Un sliding window clásico sobre el feature map solo desplaza una ventana de tamaño fijo y pasa cada parche a un clasificador. La ventana no aprende nada sobre dónde es más probable encontrar objetos en la imagen, ni ajusta su posición o tamaño. La RPN, en cambio, **aprende a proponer** y en cada posición de la ventana, predice si hay un objeto.

El feature map no es una versión comprimida de la imagen, sino una representación semántica de ella ya codifica bordes, texturas, formas y estructuras aprendidas de miles de imágenes. Cuando la RPN opera sobre ese feature map compartido, sus propuestas están informadas por esa semántica. Una posición en el feature map donde el backbone detectó un patrón visual relevante (contorno de producto, cambio de textura de etiqueta) ya tiene implícita la señal de que ahí puede haber un objeto. Eso es algo que un sliding window sobre píxeles crudos o sobre el feature map sin aprendizaje no puede aprovechar.

#### Inciso 5

La RPN utiliza anchor boxes predefinidos (9 por posición: 3 escalas × 3 relaciones de aspecto). Explique qué son los anchors y por qué la red predice deltas $(\Delta x, \Delta y, \Delta w, \Delta h)$ en lugar de coordenadas absolutas. En la decodificación
$w = w_a \cdot e^{\Delta w}$,
identifique qué representa cada símbolo y explique por qué se usa la exponencial para las dimensiones.

**Respuesta:**

Los **anchor boxes** son cajas de referencia predefinidas, centradas en cada posición de la ventana deslizante sobre el feature map. En lugar de predecir la caja directamente desde cero, la red parte de estas referencias conocidas y predice cuánto debe ajustarlas para encajar el objeto real. Para cada posición se definen $k = 9$ anchors, combinaciones de 3 escalas ($128^2$, $256^2$, $512^2$ px) y 3 proporciones de aspecto (1:1, 1:2, 2:1), cubriendo así distintos tamaños y formas de objetos sin necesidad de buscarlos explícitamente.

La red predice **deltas** $(\Delta x, \Delta y, \Delta w, \Delta h)$ en lugar de coordenadas absolutas porque los deltas son desplazamientos relativos pequeños respecto al anchor de referencia, lo que hace el problema numéricamente más estable. Predecir coordenadas absolutas significaría que la red debe aprender valores que varían enormemente entre imágenes de distintas resoluciones.

En la decodificación $w = w_a \cdot e^{\Delta w}$:

- $w_a$ es el **ancho del anchor de referencia**, el punto de partida conocido.
- $\Delta w$ es el **delta predicho por la red**, el ajuste logarítmico aprendido.
- $e^{\Delta w}$ convierte ese delta en un factor multiplicativo.

La exponencial se usa para las dimensiones (ancho y alto) porque garantiza que $w > 0$ para cualquier valor de $\Delta w$, incluyendo negativos. Si se usara una suma directa ($w = w_a + \Delta w$), un delta suficientemente negativo produciría un ancho negativo o cero, que no tiene significado geométrico.

#### Inciso 6

Faster R-CNN logra $\sim 5 , FPS$ con VGG16. La restricción de VisorShelf es procesar una imagen en menos de $500 , ms$ ($\geq 2 , FPS$). Tomando en cuenta esa restricción, ¿recomendaría Faster R-CNN para producción en el hardware de tienda? Argumente su respuesta considerando al menos dos factores distintos a la velocidad pura (p. ej., precisión en objetos pequeños y densos, facilidad de fine-tuning, ecosistema de implementación).

**Respuesta:**

Para las tiendas estándar de VisorShelf con CPU de gama media y sin GPU dedicada, **no recomendaría Faster R-CNN con VGG16 en su configuración de referencia**. Los ~5 FPS reportados son con GPU; en CPU el rendimiento cae drásticamente, muy por debajo de los 2 FPS requeridos, incumpliendo directamente la restricción de 500 ms por imagen.

1. Los anaqueles de supermercado contienen decenas de productos pequeños y adyacentes, exactamente el escenario donde la detección multi-escala de FPN marca diferencia. Descartar Faster R-CNN implica aceptar ese trade-off de precisión en el dominio de VisorShelf.

2. Faster R-CNN está ampliamente soportado en frameworks como Detectron2 y TorchVision con implementaciones estables, documentación extensa y modelos pre-entrenados en COCO listos para fine-tuning. 

La recomendación para producción en hardware sin GPU es explorar Faster R-CNN con un backbone más ligero (MobileNet o ResNet-50 con backbone cuantizado) o directamente optar por un detector de una etapa como YOLOv8n, que cumple la restricción de latencia en CPU y puede alcanzar precisión comparable con fine-tuning sobre el dataset de anaqueles.

### Pregunta 2.3

El equipo de producto presenta dos propuestas para el detector de producción de VisorShelf:

| Dimensión                           | Propuesta A: Faster R-CNN + ResNet-50 + FPN | Propuesta B: YOLOv8n (nano) fine-tuned  |
| ----------------------------------- | ------------------------------------------- | --------------------------------------- |
| Velocidad estimada                  | 8–12 FPS en GPU / $\sim 1.5$ FPS en CPU     | 60–80 FPS en GPU / $\sim 15$ FPS en CPU |
| mAP@0.5 (referencia)                | $\sim 55$ en COCO                           | $\sim 37$ en COCO                       |
| Tamaño del modelo                   | $\sim 160 , MB$                             | $\sim 6 , MB$                           |
| Manejo de objetos pequeños y densos | Alto (FPN multi-escala)                     | Moderado                                |
| Costo de fine-tuning                | Moderado (pipeline de dos etapas)           | Bajo (arquitectura simple)              |

Responda en su reporte:

#### Inciso 7

Tomando en cuenta las restricciones operativas de VisorShelf (hardware sin GPU, latencia < $500 , ms$, anaqueles densos), ¿cuál propuesta recomendaría y por qué? No se limite a comparar los números de la tabla; argumente la lógica de la decisión conectándola con las características técnicas de cada arquitectura.

**Respuesta:**

Bajo las restricciones actuales, la propuesta recomendada es **YOLOv8n**.

La razón no es solo que cumpla la restricción de velocidad (15 FPS en CPU vs. 1.5 FPS de Faster R-CNN), sino la lógica detrás de esa diferencia. Faster R-CNN es un detector de dos etapas: primero genera propuestas con la RPN, luego las clasifica y refina. Esa secuencia tiene una latencia mínima que no se puede eliminar sin cambiar de arquitectura. En CPU, ese pipeline secuencial supera con facilidad los 500 ms por imagen, incumpliendo el requisito central del sistema.

YOLOv8n, en cambio, colapsa todo el proceso en un único forward pass sobre un tensor 3D: no hay etapa de propuestas, no hay procesamiento secuencial. Toda la localización y clasificación ocurre en paralelo en una sola inferencia. Eso es lo que le permite ser operacionalmente viable en CPU de tienda.

COCO es un benchmark de 80 clases con objetos de todo tipo. En el dominio específico de VisorShelf, un dataset de anaquel con una sola clase ("producto") o pocas categorías, el fine-tuning puede cerrar esa brecha. Un modelo ligero bien ajustado al dominio frecuentemente supera a uno más potente pero genérico. Y la desventaja de manejo de objetos densos de YOLOv8n puede mitigarse ajustando el umbral de NMS adecuadamente.

#### Inciso 8

Si VisorShelf logra instalar una GPU de gama media (RTX 3060) en las tiendas flagship, ¿cambiaría su recomendación? Explique cómo ese cambio de hardware altera el trade-off entre las dos propuestas.

**Respuesta:**

Con una GPU de gama media como la RTX 3060 el cuello de botella de latencia desaparece para Faster R-CNN, ambas propuestas cumplen holgadamente el requisito de procesamiento cada 30 segundos.

Faster R-CNN + ResNet-50 + FPN pasa de ser inviable a ser competitivo, y su ventaja en manejo de objetos pequeños y densos se vuelve el factor diferenciador relevante; en el escenario donde la detección multi-escala de FPN aporta valor real sobre la cuadrícula uniforme de YOLOv8.

Con GPU, sus 60–80 FPS permiten procesar múltiples cámaras simultáneamente con un solo modelo sin saturar el hardware. Faster R-CNN a 8–12 FPS en GPU limita cuántas cámaras puede manejar en paralelo. Si la tienda flagship tiene 10 cámaras, YOLOv8 puede manejarlas todas; Faster R-CNN necesitaría hardware adicional o colas de procesamiento.

La decisión final en ese escenario depende del número de cámaras por tienda; con pocas cámaras y alta densidad de productos, Faster R-CNN + FPN; con muchas cámaras y menor densidad, YOLOv8.

#### Inciso 9

¿Qué riesgo técnico específico introduce hacer fine-tuning de Faster R-CNN con un dataset de anaqueles sin aplicar learning rate diferenciado entre el backbone y las capas nuevas? ¿Cómo se llama ese fenómeno y cómo lo mitigaría?

**Respuesta:**

El riesgo se llama **catastrophic forgetting**. Si se aplica el mismo learning rate alto al backbone pre-entrenado y a las capas nuevas del clasificador, el backbone sufre actualizaciones de gradiente demasiado grandes para el nuevo dominio, destruyendo las representaciones visuales genéricas que aprendió durante el pre-entrenamiento en ImageNet o COCO. La red "olvida" los detectores de bordes, texturas y estructuras que hacían útil el transfer learning, y debe reaprender todo desde casi cero con el dataset limitado de anaqueles.

Para mitigarlo se aplica **learning rate diferenciado por capas**:

- Las capas del backbone pre-entrenado reciben un learning rate muy bajo, que permite ajustes finos sin destruir las representaciones aprendidas.
- Las capas nuevas del clasificador y la RPN, inicializadas aleatoriamente, reciben un learning rate mayor, permitiéndoles aprender rápidamente desde cero.

Una estrategia complementaria es el **fine-tuning por fases**, primero congelar completamente el backbone y entrenar solo las capas nuevas hasta convergencia; luego descongelar gradualmente las últimas capas del backbone con learning rate bajo para ajuste fino del dominio.

## Task 3

Utilice PyTorch o TensorFlow/Keras a su elección. No se proporciona código base; usted debe construir su solución apoyándose en la documentación oficial, recursos académicos y su criterio de ingeniería. Ejecute sus experimentos en Google Colab, Kaggle Notebooks o GPU local. Entregue el enlace al notebook con todas las celdas ejecutadas y los resultados visibles. El notebook debe estar limpio, comentado y reproducible.

La evaluación considera no solo que el código funcione, sino que usted entienda cada decisión que tomó y la justifique en su reporte.

Con esto realice lo siguiente:

### 1. Preparación del Dataset:

a. **Dataset:** Descargue un subconjunto del dataset SKU110K (mínimo 500 imágenes de entrenamiento, 100 de validación, 100 de prueba). Si el dataset completo no es accesible, puede utilizar Grocery Store Dataset o Open Images V7 filtrado por categorías de productos de anaquel. Documente en su reporte la fuente exacta y el proceso de descarga.

b. **Preprocesamiento:** Asegúrese de que las anotaciones estén en formato compatible con el detector elegido (COCO JSON, YOLO `.txt`, o Pascal VOC XML). Justifique en su reporte cualquier conversión que realice y documente la distribución de clases del subconjunto utilizado.

c. **Verificación:** Visualice al menos 5 imágenes con sus bounding boxes anotados antes del entrenamiento. Incluya esas visualizaciones en su notebook.

### 2. Entrenamiento de Dos Detectores:

a. Seleccione **dos detectores pre-entrenados** de su elección (por ejemplo: Faster R-CNN con ResNet-50, SSD, DETR, RT-DETR, entre otros). Para cada uno:

i. Cargue el modelo pre-entrenado en COCO o ImageNet y adapte el cabezal de clasificación al número de clases del dataset de anaqueles. Justifique en su reporte por qué eligió esos dos modelos específicos para compararlos.

ii. Realice **fine-tuning** con las capas base inicialmente congeladas. Documente los hiperparámetros elegidos (learning rate, épocas, batch size, optimizador) y argumente por qué son razonables para este problema y tamaño de dataset.

iii. Implemente **Early Stopping** monitoreando la métrica de validación apropiada. Explique en su reporte qué métrica eligió y la lógica detrás de detener el entrenamiento anticipadamente.

iv. Guarde los pesos del mejor modelo de cada arquitectura.

### 3. Evaluación

a. Para cada modelo, calcule y registre las siguientes métricas sobre el conjunto de prueba:

i. $mAP@0.5$
ii. $mAP@0.5:0.95$
iii. Precisión y Recall para la clase de producto principal
iv. FPS promedio de inferencia sobre 50 imágenes
v. Tamaño del modelo guardado en disco (MB)

b. Visualice al menos 3 imágenes de prueba con las detecciones del mejor modelo superpuestas, mostrando las cajas predichas con su score de confianza.

## Dictamen Ejecutivo:

Redacte en su reporte un dictamen ejecutivo de 1 a 2 páginas dirigido al CTO de VisorShelf. El dictamen debe incluir:

* **Tabla comparativa** cruzando ambos modelos con todas las métricas del Paso 3.

* **Análisis de velocidad vs. precisión:** ¿Cuál modelo detecta mejor los productos en anaquel? Argumente por qué en el contexto de auditoría de stock se prefiere alta Precisión sobre alto Recall (o viceversa), y sustente su posición con los números obtenidos.

* **Recomendación final:** ¿Cuál modelo desplegaría en producción on-premise con CPU de tienda y por qué? Conecte explícitamente esta decisión con lo analizado en el Task 2.

* **Análisis de viabilidad operativa:** ¿Los FPS obtenidos son suficientes para el requerimiento de procesar una imagen cada 30 segundos? ¿Cuántas imágenes por hora podría procesar el sistema? ¿Cómo cambia esa viabilidad si la tienda instala 5 cámaras simultáneas?

* **¿Cuánto “cuesta” en MB cada punto de mAP?** Calcule la razón $MB/mAP$ para cada modelo y argumente si el modelo más pesado se justifica en el contexto de hardware limitado de VisorShelf.

* **Reflexión sobre generalización:** ¿Funcionaría el modelo igual en tiendas con diferente iluminación, cámaras de distinta resolución o categorías de productos distintas a las del dataset de entrenamiento? ¿Qué implicaciones tiene eso para la estrategia de expansión de VisorShelf?

* **Limitaciones del experimento:** Identifique al menos dos limitaciones metodológicas de su experimento que deberían resolverse antes de un despliegue real.